# Tutorial 2: Parsing with Bison

Recognize a declaration, diagnose syntax errors, and build its abstract syntax tree. Run the cells in order; save experiments before rerunning a source-writing cell.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jianwen-zhu/2026-fall-ece467-jzhu/blob/gh-pages/tutorial2.ipynb)

## Setup

Run these setup cells once, then work through the examples in order. In Colab, use a CPU runtime. On your own computer, install Make, a C compiler, and Bison first.

In [ ]:
%%bash
set -euo pipefail
missing=0
for tool in make cc bison; do command -v "$tool" >/dev/null || missing=1; done
if python3 -c 'import google.colab' 2>/dev/null; then
    if [ "$missing" -eq 1 ]; then
        apt-get -qq update
        apt-get -qq install -y build-essential bison
    fi
fi
for tool in make cc bison; do command -v "$tool"; done


In [ ]:
#@title Restore companion files
import base64, io, zipfile
from pathlib import Path

payload = "UEsDBBQAAAAIAAAAcl2KHdFRgAIAADkEAAAHAAAATElDRU5TRV1SW2/aMBR+96844qmVsm6qpk3am0lMsZbEyDFlPIbEEE8hRrFZxb/fOYG26ySkyOfy3Q6FNJC7xg7BMpb602V0hy7CXXMPj18ev4Ft7Ndv3z/19S5A44c4ut05+jEwtrLj0YXg/AAuQGdHu7vAYayHaNsE9qO14PfQdPV4sAlED/VwgZMdAy74Xazd4IYD1Ih6ujCcjB3CBL+PL/VocbiFOgTfuBrxoPXN+WiHWEfi27veBriLnYVZdduY3U8kra175gag3msLXlzs/DnCaAMaaAgjATc0/bklDa/t3h3djYHWpyQCQ9BzQAekM4Gjb92evnaydTrvehe6BFoXrtlgMVBxijQhH5/9CMH2PUMEh7onr+/qphmSfqJA4y2iQJWXzh8/OnGB7c/jgJR22mk9RjYx/rZNpAqN733f+xeyhidrHTkKPxgz2Kp3/o+dvFwPPfiIUq8S6ACn96veWqGr+x529hYY8mK89T92RqIPEQ/v6h5Ofpz4/rf5gPxLAZVamA3XAmQFK62eZSYymPEK37MENtIs1doATmhemi2oBfByCz9lmSUgfq20qCpQmslilUuBNVmm+TqT5RPMca9U+G+WhTQIahQQ4Q1KiorACqHTJT75XObSbBO2kKYkzIXSwGHFtZHpOucaVmu9UpVA+gxhS1kuNLKIQpTmAVmxBuIZH1AteZ4TFeNrVK9JH6RqtdXyaWlgqfJMYHEuUBmf5+JKhabSnMsigYwX/ElMWwpRNKOxqzrYLAWViI/jLzVSlWQjVaXR+EzQpTZvqxtZiQS4lhUFstCqSBjFiRtqAsG9UlxRKGr4cBEcofe6Em+AkAmeI1ZFy2TxdfiB/QVQSwMEFAAAAAgAAAByXUlEQaSfAgAAJAgAAA8AAABzdXBwb3J0L2FyZW5hLmO1VU1vm0AQvfMrJs4hiz9SR6oqNdRUPriSJbeHtpGiJhEisNioeEGwNHYb//fu7uzCEhM1OZSDbebjzZs3s+vTlEVZHVMYhCVl4flm4Jwa04eKxynj5xv/iS0/MmXp/VNbmbK1tDniZx1xUPjBfZZHP+GPA+LpcQwZ3XEPvelvGnCoKxqPIQqLMEr5Hl3bcBeEWbpmwh+HPLy585yD5zi/8jSGIeKFmQAkdgntGRtk+eVqKlHOKm4cCnpLGYcZBHP5kifEqul6z9LHrxkaJ756rTr9lHnN4uOWkHpJqzrT/acJkBPkKD8F6EXrUBYfvi1/LILP82uYAGlZT+DCdaGkvC4ZfLlarTBPVxZABPNH0M2BN63BhWH7YhHCBh8fGzQflGnim34ElLbI0RmBlcgmYmZlv52+fwcfG8OlMnhNjiza5HUali3kiZa8p135mHFs9TbYKaJ9A+x26538E3Liyz3tn7MVJAUQQdMjh6VEdw3k08EUEeoHug84SLUlco41q8SIRJFoE5YwdDW8PBOiPYsFptu0Rs0Q9HZgs2YDD46DmLjeYtdZXBf95wkPD4ZzoUtzwjLK1nzTnDEVEOXFvt0njIDZrJlsj+wyxWitz7WurNNHYnmtHZXxPTBbuo2KPZHeMSBPTbAtc4OWO1Hu7HZ61pEGmR8L82JdjBAar6usztH68VIQISrJVTXtq43RhxdfbJhX9MunAi3hCpEFSVHzigw4z6JLja+SQp7mDJIwzWh8ywaSZEzL0vWA7lJOFtfL78Gn+XJ19XUhbAcjeUU5KcYw7VTTChRHckqNXyemIo6G48VR0W3zeliW1P+1+06XahBII6aCSN7bp+nqYSMKAelcBfZV+sr/nmevFbzI2qCkpJTg9WdunIPzF1BLAwQUAAAACAAAAHJdg3D8V6UBAACiAwAADwAAAHN1cHBvcnQvYXJlbmEuaJVSwW7bMAy96ysI5LIZq7O7hwJBMWADghyG7GwoEh0TVShDkp24Rf99tNxmCdJDexJMv/dIvscFNWyxge12Xa/+/Nys6l9qIQVivKqpBbFxvUX4EZMVQNneKxVT6E0CHZB1vXPePFZXNXiGWwgU+YkVvFRKLQv4zZRIO3pCOFJq4fn7SwkrJyCdyHOcyNAznWEWNFs46FMtn3uu011+0ZYKCtj8Xa/hgHoinkWWcZJvNLk+YAkbHDBAExBFCogtDWR77S4YWWse26Js4UcI6FBHjDCxR5i3mUaR8iDl1OLr3gH7qHdOOhVLNXiyUMxSWf/LlUfzn28wTVin/HytlGl1eCMJnG3fvU8z4lCCGZ7wlM5CDnmf2lupDysJVdJ5aNE8iuXyd0AmZCObNj5I4dCRExvZi0HZBzGAAvijRDEFT7yPJWxbHKGPOPmZHZotFgW98wNWF56/BQSdcJNogiW9Zx8TmdwAT5TiRTKMR3jC4O/+H0eE3ZhQrmtGGN+N0iF60MZgl0Xzhczz3QQkip+NZ2rxYUdzr6uzeo8qQLVAOctG/QNQSwECFAMUAAAACAAAAHJdih3RUYACAAA5BAAABwAAAAAAAAAAAAAAgAEAAAAATElDRU5TRVBLAQIUAxQAAAAIAAAAcl1JREGknwIAACQIAAAPAAAAAAAAAAAAAACAAaUCAABzdXBwb3J0L2FyZW5hLmNQSwECFAMUAAAACAAAAHJdg3D8V6UBAACiAwAADwAAAAAAAAAAAAAAgAFxBQAAc3VwcG9ydC9hcmVuYS5oUEsFBgAAAAADAAMArwAAAEMHAAAAAA=="
with zipfile.ZipFile(io.BytesIO(base64.b64decode(payload))) as files:
    for name in files.namelist():
        path = Path(name)
        if path.is_absolute() or ".." in path.parts:
            raise ValueError("Unsafe companion path")
        if not path.exists():
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_bytes(files.read(name))
print("Companion files ready; existing files preserved.")


In [ ]:
%%bash
set -euo pipefail
mkdir -p .tutorial-2-work
cp support/arena.c support/arena.h .tutorial-2-work/


## Tokens describe parts; grammar describes structure

For `axis B := 4;`, assume the scanner supplies:

```text
AXIS IDENTIFIER ASSIGN INTEGER ';' EOF
```

`%token` declares token codes; `%start` names the start rule; the two `%%` lines delimit the grammar. The braces contain a C action run when Bison reduces that rule.

Here `yylex` supplies fixed token codes instead of scanning text. Returning zero means end of input. This lets us study parsing without a scanner.

**Predict:** which token is expected after `INTEGER`? Is another identifier after the semicolon allowed?

In [ ]:
%%writefile .tutorial-2-work/declaration.y
%{
#include <stdio.h>
#include <string.h>
int yylex(void);
void yyerror(const char *message);
static const int *input;
%}
%token AXIS IDENTIFIER ASSIGN INTEGER
%start declaration
%%
declaration: AXIS IDENTIFIER ASSIGN INTEGER ';'
             { puts("reduced declaration"); };
%%
int yylex(void) {
    int token = *input;
    if (token) ++input;
    return token;
}
void yyerror(const char *message) { fprintf(stderr, "%s\n", message); }
int main(int argc, char **argv) {
    static const int valid[] = {AXIS, IDENTIFIER, ASSIGN, INTEGER, ';', 0};
    static const int missing[] = {AXIS, IDENTIFIER, ASSIGN, INTEGER, 0};
    static const int trailing[] = {AXIS, IDENTIFIER, ASSIGN, INTEGER, ';', IDENTIFIER, 0};
    input = valid;
    if (argc == 2 && !strcmp(argv[1], "missing")) input = missing;
    else if (argc == 2 && !strcmp(argv[1], "trailing")) input = trailing;
    else if (argc != 1) return 2;
    return yyparse();
}


## Build, run, and diagnose

Bison generates C; the C compiler builds the executable. `make` expresses both steps; `-d` also generates the token header. Recipe lines start with a tab.

Run the valid input first. Then predict each failure **before** running the next cell. A rule's action may run before later input makes the whole parse fail: `yyparse()`'s return status, not the printed reduction, determines acceptance.

In [ ]:
%%writefile .tutorial-2-work/Makefile
CC = cc
CFLAGS = -std=c11 -O0 -g -Wall -Wextra
BISON = bison
.PHONY: all clean
all: declaration
declaration.c: declaration.y
	$(BISON) -d -o $@ $<
declaration: declaration.c
	$(CC) $(CFLAGS) $< -o $@
tree.c: tree.y
	$(BISON) -d -o $@ $<
tree: tree.c arena.c arena.h
	$(CC) $(CFLAGS) tree.c arena.c -o $@
clean:
	$(RM) declaration tree declaration.c declaration.h tree.c tree.h
	$(RM) actual.txt error.txt


In [ ]:
%%bash
set -euo pipefail
cd .tutorial-2-work
make
./declaration | tee actual.txt
printf 'reduced declaration\n' | diff - actual.txt

In [ ]:
%%bash
set -euo pipefail
cd .tutorial-2-work
for case in missing trailing; do
    echo "Input: $case"
    if ./declaration "$case" > actual.txt 2> error.txt; then
        echo "Expected rejection, but parsing succeeded" >&2
        exit 1
    else
        status=$?
        test "$status" -eq 1
    fi
    cat actual.txt error.txt
    grep -q 'syntax error' error.txt
done

**Diagnose:** the missing semicolon leaves an incomplete declaration; the extra identifier violates end-of-input. Both inputs must fail. Whether the trailing-input case first prints a reduction can depend on the Bison version.

**Modify:** add the missing semicolon to the `missing` array and rebuild. Predict why the failure-check cell now complains. Restore the array before running all cells again.

## Give the declaration a structure

Token codes describe categories; semantic values carry data such as the spelling `B`. The next version constructs:

```text
Axis
  Name("B")
  Integer("4")
```

`%union` lists the C value types. Typed `%token` and `%type` select them. In an action, `$1`, `$2`, etc. refer to right-hand-side values; `$$` is its result. Locate the two leaf actions and the action attaching them to a parent.

The supplied `arena_new` and `arena_copy` allocate nodes and copy retained text. Treat them as provided operations; allocator internals are outside this tutorial. Token data is borrowed; the AST must own the text it keeps.

In [ ]:
%%writefile .tutorial-2-work/tree.y
%{
#include <stdio.h>
#include "arena.h"
struct node {
    const char *kind, *text;
    struct node *left, *right;
};
static struct arena ast_arena;
static struct node *root;
int yylex(void);
void yyerror(const char *message);
%}
%union { const char *text; struct node *node; }
%token AXIS ASSIGN
%token <text> IDENTIFIER INTEGER
%type <node> name extent declaration
%start declaration
%%
name: IDENTIFIER {
    $$ = arena_new(&ast_arena, sizeof *$$);
    $$->kind = "Name";
    $$->text = arena_copy(&ast_arena, $1);
};
extent: INTEGER {
    $$ = arena_new(&ast_arena, sizeof *$$);
    $$->kind = "Integer";
    $$->text = arena_copy(&ast_arena, $1);
};
declaration: AXIS name ASSIGN extent ';' {
    $$ = arena_new(&ast_arena, sizeof *$$);
    $$->kind = "Axis";
    $$->left = $2;
    $$->right = $4;
    root = $$;
};
%%
int yylex(void) {
    static unsigned next;
    static const int tokens[] = {AXIS, IDENTIFIER, ASSIGN, INTEGER, ';', 0};
    int token = tokens[next];
    if (token) ++next;
    if (token == IDENTIFIER) yylval.text = "B";
    if (token == INTEGER) yylval.text = "4";
    return token;
}
void yyerror(const char *message) { fprintf(stderr, "%s\n", message); }
int main(void) {
    int status = yyparse();
    if (!status)
        printf("%s(%s(%s), %s(%s))\n", root->kind,
               root->left->kind, root->left->text,
               root->right->kind, root->right->text);
    arena_destroy(&ast_arena);
    return status;
}


In [ ]:
%%bash
set -euo pipefail
cd .tutorial-2-work
make tree
./tree | tee actual.txt
printf 'Axis(Name(B), Integer(4))\n' | diff - actual.txt

**Predict–modify–diagnose:** change the identifier spelling in `yylex` from `B` to `C` and rebuild. Which printed field should change? The grammar is unchanged: names are data, not separate grammar rules. Restore `B` afterwards.

Notice that `:=` and `;` have no AST nodes. They organize the source; the AST retains the declaration's meaning. The toy printer runs only after successful parsing. The project must similarly not publish a partial AST on failure.

## Connect this to Project Stage 2

Keep the project's supplied Makefile and driver, not the toy's `main` or fixed token arrays.

| Toy | Stage 2 starter |
| --- | --- |
| Fixed tokens from `yylex` | Supplied `token_adapter.c` provides tokens and locations |
| Borrowed string semantic values | Borrowed `const struct token *`; read the token's `text` |
| Nodes and copied text in `ast_arena` | Your native AST in `parser_arena`, root in `parsed_ast` |
| Small tree printer | Separate `ast_to_json` checkpoint writer |

JSON is the saved AST format, not the representation grammar actions must construct. The project implements serialization separately.

From the extracted Stage 2 package, the familiar build/check loop is:

```sh
make -C 02_parser
make -C 02_parser check-starter
# After implementing your parser:
make -C 02_parser check
```

`check-starter` tests the untouched scaffold, not a solution. `check` is expected to fail until you implement the stage. Supplied token files let you start without a working Stage 1 scanner.

## Recap and questions

Check your understanding:
- Why valid individual tokens can form invalid syntax.
- Why a reduction does not guarantee whole-input acceptance.
- How `$2` and `$4` become children of the declaration.
- Why an undeclared but syntactically valid name is Stage 3's concern.

### Try another example

Sketch the AST for a declaration with another name and extent; mark which tokens become nodes and which disappear. No conflict reports, parser-state reports, or extra tooling are required.

The final cell removes only demonstration build products; source files remain for inspection.

In [ ]:
%%bash
set -euo pipefail
cd .tutorial-2-work
make clean
printf 'Tutorial 2 demonstrations passed; generated files removed.\n'